# Sum of Square Spectral Amplification Resource Estimation

This notebook demonstrates how to use QDK Chemistry to construct a Sum of Squares Spectral Amplification (SOSSA) block-encoding circuit and perform resource estimation using QRE v3.
The goal is to enable users to understand and further improve on the fault-tolerant resource requirements of state-of-the-art quantum algorithms.

In addition to [installing `qdk-chemistry`](https://github.com/microsoft/qdk-chemistry/blob/main/INSTALL.md), you will need to install the `jupyter` and `qre` extras to run this notebook:

```bash
pip install 'qdk-chemistry[jupyter,qre]'
```

In [1]:
# Reduce logging output for demo
from qdk_chemistry.utils import Logger
Logger.set_global_level(Logger.LogLevel.off)

## Setup a factorized Hamiltonian for H2

**No DFTHC factorization required.** This notebook uses only `(N, R, B, C, b_rot, b_coeff, lambda_eff)` parameters to generate fake data of correct size.

In [2]:
from math import sqrt

import numpy as np
import pandas as pd

from qdk_chemistry.algorithms.controlled_circuit_mapper.sossa_mapper import (
    InnerPrepareMapper,
    OuterPrepareMapper,
    SelectMapper,
    SOSSAMapper,
)
from qdk_chemistry.algorithms.hamiltonian_unitary_builder.block_encoding.sossa import SOSSABuilder
from qdk_chemistry.data import BasisSet, FactorizedHamiltonianContainer, Orbitals, OrbitalType, Shell
from qdk_chemistry.data.controlled_unitary import ControlledUnitary
from qdk_chemistry.data.unitary_representation.containers.sossa import SOSSAContainer

# H2-like DFTHC factorized Hamiltonian: N=2 orbitals, R=1, B=1, C=1
N, R, B, C = 2, 1, 1, 1

h1 = np.array([
    [0.3, 0.1],
    [0.1, -0.2],
])
basis_vectors = np.array([[[1.0 / sqrt(2), 1.0 / sqrt(2)]]])  # [R, B, N]
two_body_weights = np.array([[[0.15]]])                         # [R, B, C]
identity_weight = np.array([[0.08]])                            # [R, C]

# Build FactorizedHamiltonianContainer
shells = [Shell(0, OrbitalType.S, np.array([1.0]), np.array([1.0])) for _ in range(N)]
basis_set = BasisSet("test-basis", shells)
orbitals = Orbitals(np.eye(N), None, None, basis_set)
inactive_fock = np.zeros((N, N))

fh = FactorizedHamiltonianContainer(
    h1, basis_vectors.flatten(),
    two_body_weights.flatten(), identity_weight,
    R, B, C,
    orbitals, 0.0, inactive_fock,
)
print(f"Factorized Hamiltonian: N={N}, R={R}, B={B}, C={C}")

Factorized Hamiltonian: N=2, R=1, B=1, C=1


## Generate SOSSA Circuit

In [3]:
# Step 1: SOSSABuilder → UnitaryRepresentation
builder = SOSSABuilder(quantum_walk=True)
unitary_rep = builder.run(fh)
container = unitary_rep.get_container()

# Step 2: SOSSAMapper → Circuit
controlled_unitary = ControlledUnitary(unitary=unitary_rep, control_indices=[0])
mapper = SOSSAMapper(
    outer_prepare_mapper=OuterPrepareMapper(algorithm="dense_pure"),
    inner_prepare_mapper=InnerPrepareMapper(algorithm="direct"),
    select_mapper=SelectMapper(multiplexed_rotation="direct"),
)
circuit = mapper.run(controlled_unitary)
lambda_sos = container.normalization
print(f"SOSSA normalization Λ = {lambda_sos:.6f}")
print(f"Circuit built successfully")

SOSSA normalization Λ = 1.122596
Circuit built successfully


## Verify QPE simulation

In [4]:
from qdk_chemistry.algorithms.phase_estimation.iterative_phase_estimation import IterativePhaseEstimation
from qdk_chemistry.data import AlgorithmRef
from qdk_chemistry.data.circuit import Circuit, QsharpFactoryData
from qdk_chemistry.utils.qsharp import QSHARP_UTILS

# Hartree-Fock state |1100⟩ as trial state (2 electrons in lowest orbitals)
num_system_qubits = 2 * N
hf_bitstring = [1, 1] + [0] * (num_system_qubits - 2)
state_prep_params = {
    "rowMap": list(range(num_system_qubits - 1, -1, -1)),
    "stateVector": [0.0] * (2**num_system_qubits),
    "expansionOps": [],
    "numQubits": num_system_qubits,
}
# Set HF amplitude: |1100⟩ in big-endian = index 12
hf_index = sum(b << (num_system_qubits - 1 - i) for i, b in enumerate(hf_bitstring))
state_prep_params["stateVector"][hf_index] = 1.0

qsharp_factory = QsharpFactoryData(
    program=QSHARP_UTILS.StatePreparation.MakeStatePreparationCircuit,
    parameter=state_prep_params,
)
qsharp_op = QSHARP_UTILS.StatePreparation.MakeStatePreparationOp(state_prep_params)
state_prep = Circuit(qsharp_factory=qsharp_factory, qsharp_op=qsharp_op)

# Run IQPE with SOSSA walk operator
num_bits = 5
iqpe = IterativePhaseEstimation(shots_per_bit=5)
iqpe.settings().set("qpe_circuit_builder", AlgorithmRef(
    "qpe_circuit_builder", "qdk_iterative",
    num_bits=num_bits,
    controlled_circuit_mapper=AlgorithmRef(
        "controlled_circuit_mapper", "sossa",
        outer_prepare_algorithm="dense_pure",
        inner_prepare_algorithm="direct",
        select_algorithm="direct",
    ),
    unitary_builder=AlgorithmRef("hamiltonian_unitary_builder", "sossa", quantum_walk=True),
))
iqpe.settings().set(
    "circuit_executor",
    AlgorithmRef("circuit_executor", "qdk_sparse_state_simulator"),
)

result = iqpe.run(
    state_preparation=state_prep,
    factorized_hamiltonian=fh,
)
print(f"QPE phase: {result.phase_fraction:.6f}")
print(f"QPE raw energy (Λ·cos 2πφ): {result.raw_energy:.6f}")
print(f"Energy gap estimate: {lambda_sos - result.raw_energy:.6f}")

QPE phase: 0.000000
QPE raw energy (Λ·cos 2πφ): 1.122596
Energy gap estimate: 0.000000


## Resource Estimation

In [7]:
from qdk_chemistry.algorithms.phase_estimation.circuit_builder.iterative_builder import QdkIterativeQpeCircuitBuilder

# Build the full iQPE circuit for the most expensive iteration (iteration 0, power=2^(n-1))
num_bits_re = 10

iqpe_builder = QdkIterativeQpeCircuitBuilder(
    num_bits=num_bits_re,
    num_iteration=0,  # most expensive iteration: power = 2^(num_bits-1)
    controlled_circuit_mapper=AlgorithmRef(
        "controlled_circuit_mapper", "sossa",
        outer_prepare_algorithm="dense_pure",
        inner_prepare_algorithm="direct",
        select_algorithm="direct",
    ),
    unitary_builder=AlgorithmRef("hamiltonian_unitary_builder", "sossa", quantum_walk=True),
)

# Build the full iQPE iteration circuit (state prep + controlled walk + phase kickback)
iqpe_circuits = iqpe_builder.run(
    state_preparation=state_prep,
    factorized_hamiltonian=fh,
)
iqpe_circuit = iqpe_circuits[0]

# Resource estimation via qsharp.estimate (QRE)
qpe_estimate = iqpe_circuit.estimate()
df_qpe = pd.DataFrame(
    qpe_estimate.logical_counts.items(),
    columns=["Logical Estimate", "Counts"],
)
print(f"iQPE logical resource counts ({num_bits_re}-bit precision, most expensive iteration, power={2**(num_bits_re-1)}):")
display(df_qpe)

iQPE logical resource counts (10-bit precision, most expensive iteration, power=512):


,Logical Estimate,Counts
0,numQubits,15
1,tCount,4096
2,rotationCount,15360
3,rotationDepth,15359
4,cczCount,61952
5,ccixCount,0
6,measurementCount,2561


In [8]:
# Single walk-step resource estimate (no state prep / phase kickback)
df_walk = pd.DataFrame(
    circuit.estimate().logical_counts.items(),
    columns=["Logical Estimate", "Counts"],
)
print("Walk operator logical resource counts (single step):")
display(df_walk)

Walk operator logical resource counts (single step):


,Logical Estimate,Counts
0,numQubits,14
1,tCount,8
2,rotationCount,30
3,rotationDepth,29
4,cczCount,121
5,ccixCount,0
6,measurementCount,5
